# ClearBank — Análise Financeira

Desafio final: ler e validar `transacoes.csv`, gerar métricas mensais, sinalizar transações suspeitas (> R$ 10.000), exibir relatório no terminal e exportar `relatorio.json`.

**Requisitos opcionais:** `analise_pandas.py` (RO1) e `grafico.png` (RO2).

Execute as células **em ordem**. A última célula é a execução principal.


In [1]:
import csv
import json
from datetime import date, datetime

LIMITE_SUSPEITO = 10000.00
TIPOS_VALIDOS = {"credito", "debito"}


def formatar_brl(valor: float) -> str:
    """Formata valor no padrão brasileiro para exibição no terminal."""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


print(f"LIMITE_SUSPEITO = {LIMITE_SUSPEITO}")


LIMITE_SUSPEITO = 10000.0


## Leitura do CSV

In [2]:
def ler_transacoes(caminho: str = "transacoes.csv") -> list[dict]:
    """Lê o CSV com csv.DictReader. Retorna [] se o arquivo não existir."""
    try:
        with open(caminho, encoding="utf-8", newline="") as arquivo:
            return list(csv.DictReader(arquivo))
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {caminho}")
        return []


# teste rápido
_amostra = ler_transacoes()
print(f"Linhas brutas lidas: {len(_amostra)}")
if _amostra:
    print("Colunas:", list(_amostra[0].keys()))


Linhas brutas lidas: 23
Colunas: ['id', 'data', 'cliente_id', 'tipo', 'valor', 'descricao', 'categoria']


## Validação e limpeza

In [3]:
def validar_transacao(linha: dict) -> dict | None:
    """Valida uma linha. Retorna o registro limpo ou None (descarte silencioso)."""
    id_texto = (linha.get("id") or "").strip()
    if not id_texto.isdigit():
        return None

    cliente_id = (linha.get("cliente_id") or "").strip()
    if not cliente_id:
        return None

    data_texto = (linha.get("data") or "").strip()
    try:
        data = datetime.strptime(data_texto, "%Y-%m-%d")
    except ValueError:
        return None

    tipo = (linha.get("tipo") or "").strip().lower()
    if tipo not in TIPOS_VALIDOS:
        return None

    valor_texto = (linha.get("valor") or "").strip()
    try:
        valor = float(valor_texto)
    except ValueError:
        return None
    if valor <= 0:
        return None

    return {
        "id": int(id_texto),
        "data": data,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": (linha.get("descricao") or "").strip(),
        "categoria": (linha.get("categoria") or "").strip(),
    }


# teste rápido
_ok = sum(1 for l in _amostra if validar_transacao(l) is not None)
_bad = len(_amostra) - _ok
print(f"Válidas (teste): {_ok} | Inválidas (teste): {_bad}")


Válidas (teste): 16 | Inválidas (teste): 7


## Relatório mensal e suspeitas

In [4]:
def gerar_relatorio(transacoes: list[dict]) -> dict:
    """Agrupa por mês, calcula métricas, identifica suspeitas e o período."""
    resumo_mensal: dict[str, dict] = {}
    suspeitas: list[dict] = []

    for t in transacoes:
        mes = t["data"].strftime("%Y-%m")
        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "valores": [],
            }
        bucket = resumo_mensal[mes]
        bucket["quantidade"] += 1
        bucket["valores"].append(t["valor"])
        if t["tipo"] == "credito":
            bucket["total_credito"] += t["valor"]
        else:
            bucket["total_debito"] += t["valor"]

        if t["valor"] > LIMITE_SUSPEITO:
            suspeitas.append(
                {
                    "id": t["id"],
                    "cliente_id": t["cliente_id"],
                    "data": t["data"].strftime("%Y-%m-%d"),
                    "valor": t["valor"],
                }
            )

    for mes, bucket in resumo_mensal.items():
        valores = bucket.pop("valores")
        credito = bucket["total_credito"]
        debito = bucket["total_debito"]
        bucket["saldo"] = round(credito - debito, 2)
        bucket["media"] = round(sum(valores) / len(valores), 2)
        bucket["maior_valor"] = max(valores)
        bucket["menor_valor"] = min(valores)
        bucket["total_credito"] = round(credito, 2)
        bucket["total_debito"] = round(debito, 2)

    datas = [t["data"] for t in transacoes]
    inicio = min(datas)
    fim = max(datas)
    periodo = {
        "inicio": inicio.strftime("%Y-%m-%d"),
        "fim": fim.strftime("%Y-%m-%d"),
        "dias": (fim - inicio).days,
    }

    return {
        "resumo_mensal": dict(sorted(resumo_mensal.items())),
        "suspeitas": suspeitas,
        "periodo": periodo,
    }


print("gerar_relatorio pronta")


gerar_relatorio pronta


## Exibição no terminal

In [5]:
def exibir_relatorio(
    total_lidas: int,
    total_validas: int,
    total_invalidas: int,
    relatorio: dict,
) -> None:
    """Imprime o relatório formatado no terminal."""
    periodo = relatorio["periodo"]
    print("===== LIMPEZA DOS DADOS =====")
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas: {total_validas}")
    print(f"Linhas inválidas: {total_invalidas}")
    print()
    print(f"Período analisado: {periodo['inicio']} -> {periodo['fim']}")
    print(f"Dias no período: {periodo['dias']}")
    print()
    print("===== RELATÓRIO MENSAL =====")
    for mes, m in relatorio["resumo_mensal"].items():
        print(f"\nMês: {mes}")
        print(f"  Transações: {m['quantidade']}")
        print(f"  Total crédito: {formatar_brl(m['total_credito'])}")
        print(f"  Total débito:  {formatar_brl(m['total_debito'])}")
        print(f"  Saldo:         {formatar_brl(m['saldo'])}")
        print(f"  Média:         {formatar_brl(m['media'])}")
        print(f"  Maior valor:   {formatar_brl(m['maior_valor'])}")
        print(f"  Menor valor:   {formatar_brl(m['menor_valor'])}")

    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    if not relatorio["suspeitas"]:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for s in relatorio["suspeitas"]:
            print(
                f"ID: {s['id']} | Cliente: {s['cliente_id']} | "
                f"Data: {s['data']} | Valor: {formatar_brl(s['valor'])}"
            )


print("exibir_relatorio pronta")


exibir_relatorio pronta


## Exportação JSON

In [6]:
def salvar_json(
    total_validas: int,
    total_invalidas: int,
    relatorio: dict,
    caminho: str = "relatorio.json",
) -> None:
    """Exporta o relatório em JSON."""
    payload = {
        "gerado_em": date.today().isoformat(),
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "periodo": relatorio["periodo"],
        "resumo_mensal": relatorio["resumo_mensal"],
        "suspeitas": relatorio["suspeitas"],
    }
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(payload, arquivo, ensure_ascii=False, indent=2)
    print(f"Relatório salvo em {caminho}")


print("salvar_json pronta")


salvar_json pronta


## Gráfico (RO2)

In [7]:
def gerar_grafico_saldo(resumo_mensal: dict, caminho: str = "grafico.png") -> None:
    """RO2 — gráfico de barras com saldo mensal."""
    import matplotlib.pyplot as plt

    meses = list(resumo_mensal.keys())
    saldos = [resumo_mensal[m]["saldo"] for m in meses]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(meses, saldos, color="#2F6F4E")
    ax.set_title("Saldo mensal (crédito − débito)")
    ax.set_xlabel("Mês")
    ax.set_ylabel("Saldo (R$)")
    ax.axhline(0, color="#333333", linewidth=0.8)
    fig.tight_layout()
    fig.savefig(caminho, dpi=120)
    plt.close(fig)
    print(f"Gráfico salvo em {caminho}")


print("gerar_grafico_saldo pronta")


gerar_grafico_saldo pronta


## Célula de execução principal

In [8]:
# ===== CÉLULA DE EXECUÇÃO PRINCIPAL =====
brutas = ler_transacoes("transacoes.csv")
total_lidas = len(brutas)
validas = []
for linha in brutas:
    limpa = validar_transacao(linha)
    if limpa is not None:
        validas.append(limpa)

total_validas = len(validas)
total_invalidas = total_lidas - total_validas

if total_lidas == 0:
    print("Sem dados para processar.")
elif total_validas == 0:
    print("===== LIMPEZA DOS DADOS =====")
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas: {total_validas}")
    print(f"Linhas inválidas: {total_invalidas}")
    print("Nenhuma transação válida para gerar relatório.")
else:
    relatorio = gerar_relatorio(validas)
    exibir_relatorio(total_lidas, total_validas, total_invalidas, relatorio)
    salvar_json(total_validas, total_invalidas, relatorio)
    gerar_grafico_saldo(relatorio["resumo_mensal"])


===== LIMPEZA DOS DADOS =====
Total de linhas lidas: 23
Linhas válidas: 16
Linhas inválidas: 7

Período analisado: 2026-01-05 -> 2026-03-28
Dias no período: 82

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 5
  Total crédito: R$ 4.700,00
  Total débito:  R$ 720,40
  Saldo:         R$ 3.979,60
  Média:         R$ 1.084,08
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 89,90

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 17.800,00
  Total débito:  R$ 595,00
  Saldo:         R$ 17.205,00
  Média:         R$ 3.679,00
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 75,00

Mês: 2026-03
  Transações: 6
  Total crédito: R$ 16.400,00
  Total débito:  R$ 809,90
  Saldo:         R$ 15.590,10
  Média:         R$ 2.868,32
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 99,90

===== TRANSAÇÕES SUSPEITAS =====
ID: 8 | Cliente: CLI003 | Data: 2026-02-14 | Valor: R$ 15.000,00
ID: 14 | Cliente: CLI003 | Data: 2026-03-18 | Valor: R$ 12.500,00

Relatório salvo em C:\Users\milen\clearba

## RO1 — pandas

A versão alternativa está em `analise_pandas.py` (não mistura com a solução nativa).

```bash
pip install pandas matplotlib
python analise_pandas.py
```

O script recalcula as métricas com `groupby` e compara com `relatorio.json`.
